In [2]:
from __future__ import annotations

import json
import random
import shutil
from pathlib import Path

In [3]:
# -----------------------------
# CONFIG
# -----------------------------
PROJECT_DIR = Path(r"C:\Projects\PoseEstimation")

JSON_DIR = PROJECT_DIR / "auto_pose_output" / "json_labels"
DATASET_DIR = PROJECT_DIR / "dataset_pose"

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

RANDOM_SEED = 42

# For deciding whether an auto-labeled keypoint counts as visible
KPT_CONF_THRESHOLD = 0.20

KEYPOINT_NAMES = [
    "nose",
    "left_eye",
    "right_eye",
    "left_ear",
    "right_ear",
    "left_shoulder",
    "right_shoulder",
    "left_elbow",
    "right_elbow",
    "left_wrist",
    "right_wrist",
    "left_hip",
    "right_hip",
    "left_knee",
    "right_knee",
    "left_ankle",
    "right_ankle",
]

In [4]:
def make_dirs(base: Path) -> None:
    for split in ["train", "val", "test"]:
        (base / "images" / split).mkdir(parents=True, exist_ok=True)
        (base / "labels" / split).mkdir(parents=True, exist_ok=True)


def clamp01(x: float) -> float:
    return max(0.0, min(1.0, x))


def xyxy_to_xywhn(x1: float, y1: float, x2: float, y2: float, img_w: int, img_h: int):
    cx = ((x1 + x2) / 2.0) / img_w
    cy = ((y1 + y2) / 2.0) / img_h
    w = (x2 - x1) / img_w
    h = (y2 - y1) / img_h
    return clamp01(cx), clamp01(cy), clamp01(w), clamp01(h)


def safe_name_from_json(json_path: Path) -> str:
    return json_path.stem


def build_yolo_pose_line(det: dict, img_w: int, img_h: int) -> str | None:
    """
    Convert one detection dict from your JSON output into one YOLO pose label line.
    """
    bbox = det.get("bbox_xyxy")
    if not bbox or len(bbox) != 4:
        return None

    x1, y1, x2, y2 = bbox
    cx, cy, w, h = xyxy_to_xywhn(x1, y1, x2, y2, img_w, img_h)

    parts = ["0", f"{cx:.6f}", f"{cy:.6f}", f"{w:.6f}", f"{h:.6f}"]  # class 0 = person

    keypoints = det.get("keypoints", {})
    for name in KEYPOINT_NAMES:
        kp = keypoints.get(name)

        if kp is None:
            # missing keypoint
            parts.extend(["0.000000", "0.000000", "0"])
            continue

        x = kp.get("x", 0.0)
        y = kp.get("y", 0.0)
        conf = kp.get("confidence", None)

        # Normalize x,y
        xn = clamp01(float(x) / img_w)
        yn = clamp01(float(y) / img_h)

        # Visibility flag
        # 2 = visible/labeled, 0 = not labeled
        if conf is None:
            v = 2
        elif float(conf) >= KPT_CONF_THRESHOLD:
            v = 2
        else:
            v = 0

        parts.extend([f"{xn:.6f}", f"{yn:.6f}", str(v)])

    return " ".join(parts)

In [5]:
json_files = sorted(JSON_DIR.glob("*.json"))
print(f"Found {len(json_files)} JSON label files.")

if not json_files:
    raise ValueError("No JSON files found. Check JSON_DIR.")

Found 1209 JSON label files.


In [6]:
# Shuffle and split
random.seed(RANDOM_SEED)
json_files_shuffled = json_files[:]
random.shuffle(json_files_shuffled)

n = len(json_files_shuffled)
n_train = int(n * TRAIN_RATIO)
n_val = int(n * VAL_RATIO)
n_test = n - n_train - n_val

train_files = json_files_shuffled[:n_train]
val_files = json_files_shuffled[n_train:n_train + n_val]
test_files = json_files_shuffled[n_train + n_val:]

print("Split sizes:")
print("train:", len(train_files))
print("val:  ", len(val_files))
print("test: ", len(test_files))

make_dirs(DATASET_DIR)

Split sizes:
train: 846
val:   181
test:  182


In [7]:
from PIL import Image

def process_split(files: list[Path], split: str) -> int:
    count = 0

    for json_path in files:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        image_path = Path(data["image_path"])

        if not image_path.exists():
            print(f"Skipping missing image: {image_path}")
            continue

        detections = data.get("detections", [])
        if not detections:
            # No detections -> skip for pose training
            continue

        # Use original image
        img = Image.open(image_path)
        img_w, img_h = img.size

        # Build label lines, one per detected person
        label_lines = []
        for det in detections:
            line = build_yolo_pose_line(det, img_w, img_h)
            if line is not None:
                label_lines.append(line)

        if not label_lines:
            continue

        stem = safe_name_from_json(json_path)
        out_img_path = DATASET_DIR / "images" / split / f"{stem}{image_path.suffix.lower()}"
        out_lbl_path = DATASET_DIR / "labels" / split / f"{stem}.txt"

        shutil.copy2(image_path, out_img_path)

        with open(out_lbl_path, "w", encoding="utf-8") as f:
            f.write("\n".join(label_lines) + "\n")

        count += 1

    return count

In [8]:
train_count = process_split(train_files, "train")
val_count = process_split(val_files, "val")
test_count = process_split(test_files, "test")

print("Actually written:")
print("train:", train_count)
print("val:  ", val_count)
print("test: ", test_count)

Actually written:
train: 843
val:   181
test:  181


In [9]:
yaml_text = f"""
path: {DATASET_DIR.as_posix()}
train: images/train
val: images/val
test: images/test

kpt_shape: [17, 3]
flip_idx: [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]

names:
  0: person
""".strip()

yaml_path = DATASET_DIR / "data.yaml"
yaml_path.write_text(yaml_text, encoding="utf-8")

print("Saved:", yaml_path)
print(yaml_text)

Saved: C:\Projects\PoseEstimation\dataset_pose\data.yaml
path: C:/Projects/PoseEstimation/dataset_pose
train: images/train
val: images/val
test: images/test

kpt_shape: [17, 3]
flip_idx: [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]

names:
  0: person


In [10]:
# Quick sanity check
for split in ["train", "val", "test"]:
    img_count = len(list((DATASET_DIR / "images" / split).glob("*")))
    lbl_count = len(list((DATASET_DIR / "labels" / split).glob("*.txt")))
    print(split, "images:", img_count, "labels:", lbl_count)

train images: 843 labels: 843
val images: 181 labels: 181
test images: 181 labels: 181


In [11]:
from ultralytics import YOLO

In [12]:
# Load pretrained pose model
model = YOLO("yolov8n-pose.pt")

In [ ]:
results = model.train(
    data=str(DATASET_DIR / "data.yaml"),
    epochs=30,
    imgsz=640,
    batch=16,
    device="cpu",
    project=str(PROJECT_DIR / "runs_pose"),
    name="yolov8n_pose_finetune",
    pretrained=True,
)

Ultralytics 8.4.42  Python-3.13.5 torch-2.10.0+cu130 CPU (Intel Core Ultra 7 256V)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Projects\PoseEstimation\dataset_pose\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_pose_finetune, nbs=64, nms=False, opset=None, optimize=False, optimize

In [ ]:
# Validate on val set
metrics = model.val(
    data=str(DATASET_DIR / "data.yaml"),
    split="val"
)

metrics

In [ ]:
# Test on test set
test_metrics = model.val(
    data=str(DATASET_DIR / "data.yaml"),
    split="test"
)

test_metrics

In [ ]:
# Run inference on a few test images with the fine-tuned model
best_model_path = PROJECT_DIR / "runs_pose" / "yolov8n_pose_finetune" / "weights" / "best.pt"
best_model = YOLO(str(best_model_path))

sample_test_images = list((DATASET_DIR / "images" / "test").glob("*"))[:5]
sample_test_images

In [ ]:
from IPython.display import Image as IPImage, display

pred_dir = PROJECT_DIR / "sample_predictions"
pred_dir.mkdir(exist_ok=True)

for img_path in sample_test_images:
    res = best_model.predict(source=str(img_path), conf=0.25, save=False, verbose=False)
    plotted = res[0].plot()
    out_path = pred_dir / img_path.name

    import cv2
    cv2.imwrite(str(out_path), plotted)

    display(IPImage(filename=str(out_path)))